In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

DATASET = "../dataset/splitted"
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6

CLASS_NAMES = [
    "Gilmore", "Golden Boy", "Golden Boy x Gilmore",
    "Golden Boy x Harold Brown", "Harold Brown", "Kearny"
]

In [8]:
transform_resnet = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5]),
])

test_loader_resnet = DataLoader(
    datasets.ImageFolder(f"{DATASET}/test", transform=transform_resnet),
    batch_size=16
)
test_loader_vit = DataLoader(
    datasets.ImageFolder(f"{DATASET}/test", transform=transform_vit),
    batch_size=16
)

In [9]:
def evaluate_model(model, model_name, test_loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            preds = outputs.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # --- Classification Report ---
    print(f"\n{'='*60}")
    print(f"  {model_name} - Classification Report")
    print(f"{'='*60}")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

    # --- Confusion Matrix ---
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    filename = f"{model_name.lower().replace(' ', '_')}_confusion_matrix.png"
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"✅ Saved: {filename}")

In [ ]:
resnet = models.resnet50(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, NUM_CLASSES)
resnet.load_state_dict(torch.load("./resnet/resnet50_best.pt", map_location=DEVICE))
resnet = resnet.to(DEVICE)

vit = models.vit_b_16(weights=None)
vit.heads.head = nn.Linear(vit.heads.head.in_features, NUM_CLASSES)
vit.load_state_dict(torch.load("./vit/vit_best.pt", map_location=DEVICE))
vit = vit.to(DEVICE)

evaluate_model(resnet, "ResNet50", test_loader_resnet)
evaluate_model(vit,    "ViT",      test_loader_vit)

FileNotFoundError: [Errno 2] No such file or directory: 'resnet50_best.pt'